# Хранение базы

Как база лежит на диске: дерево тем в три уровня, имена, формат ноутбуков, метаданные секций, теги, жизненный цикл правок и версионирование через git. Читать, когда нужно понять формат файлов базы, поля `metadata.kb` или что происходит с секцией после правки.

## Термины

| Термин | В API | Что это | На диске |
|---|---|---|---|
| База знаний | `kb` | всё дерево | `jupyter_data/kb/` |
| Тема | `topic` | узел дерева: 1 — область, 2 — тема, 3 — подтема | папка + `topic.ipynb` |
| Статья | `article` | текст по узкому вопросу внутри темы | ноутбук `<имя>.ipynb` |
| Секция | `section` | кусок статьи с заголовком; единица записи, чтения, индексации | Markdown-ячейка |
| Шапка | `head` | секция с названием и кратким описанием темы/статьи | ячейка `topic.ipynb` / первая ячейка статьи |
| Путь | `path` | адрес темы, статьи или секции | — |
| Тег | `tag` | связь секции с другими местами вне дерева | `metadata.kb.tags` |
| Индекс | `index` | список статей «путь → название» | в памяти |
| Индексатор | `indexer` | обслуживание через `/mcp/kb-admin`, приводит статьи в порядок; в первую очередь сильная внешняя модель (см. `development/mcp-kb/overview#decisions`) | — |
| Потребитель | — | ИИ, работающий через `/mcp/kb` | — |

Слова «папка», «файл», «ячейка», «ноутбук» в API не используются.

## Дерево и имена

```
jupyter_data/kb/
└── coding/                    ОБЛАСТЬ (уровень 1)
    ├── topic.ipynb            шапка области
    ├── python/                ТЕМА (уровень 2)
    │   ├── topic.ipynb
    │   ├── asyncio.ipynb      статья
    │   └── packaging/         ПОДТЕМА (уровень 3)
    │       ├── topic.ipynb
    │       └── wheels.ipynb
```

| Уровень | Может содержать |
|---|---|
| корень `kb/` | только области |
| 1 — область | шапку + только темы (статей нет) |
| 2 — тема | шапку, статьи, подтемы |
| 3 — подтема | шапку, статьи |

- Глубже 3-го уровня тем нет.
- Скрытые папки и файлы (`.git/`, `.ipynb_checkpoints/`) в базу не входят.
- Имена тем и статей — только `a-z`, `0-9`, `-`; русское название — в шапке. Имя `topic` зарезервировано.

## Формат статьи и шапки темы

Все файлы базы — ноутбуки `.ipynb` (nbformat 4.5+) **только из Markdown-ячеек**; `code` и `raw` запрещены. Через MCP нельзя создать файлы `.py` или ячейки кода; фрагменты кода — как текст в блоках ```` ``` ```` внутри секций, они не исполняются. Наши метаданные — в ключе `metadata.kb` ячейки.

**Метаданные ноутбука** (`kernelspec`, `language_info` и прочее в `metadata` верхнего уровня) в `version` и хешах секций не учитываются (`development/mcp-kb/admin-api#version`).
- Запись из веба (сохраняет JupyterLab) — плагин их не трогает.
- Запись через MCP или индексатор: есть `kernelspec` — остаётся как есть; нет (новая статья или пустые `metadata`) — плагин пишет то, что записало бы ядро при открытии, поэтому JupyterLab не считает нетронутую статью изменённой (`development/mcp-kb/setup#lab-settings`):
  - `kernelspec` — ядро по умолчанию из менеджера ядер Jupyter (имя, отображаемое имя, язык);
  - `language_info` (для Python-ядер) — плагин один раз на старте спрашивает его у интерпретатора ядра (`argv[0]` kernelspec) коротким подпроцессом: `IPythonKernel.language_info` из ipykernel — именно это ядро и записывает. Не удалось (нет ipykernel, ядро не Python) — пишется только `kernelspec`.

**Статья**
- Первая секция — шапка: `# Название` + краткое описание, `id = head`. Главный источник описания статьи.
- Остальные — секции знаний: первая строка `## Заголовок`; `###` и ниже — подзаголовки внутри секции. Jupyter показывает заголовки в оглавлении и даёт по ним свёртку.
- Заголовки уникальны в статье (без учёта регистра); через MCP дубль не создать (`duplicate_title`), дубли из веба исправляет индексатор.
- Секции без заголовка (например, человек стёр его в вебе) потребитель не видит и править не может; их чинит индексатор.

**Шапка темы** — `topic.ipynb` с одной секцией: `# Название темы` + «для чего эта тема». Есть у темы любого уровня.

**Описания (шапки)**
- Тема и статья создаются через MCP сразу с шапкой (название + краткое описание).
- Шапки индексируются как секции знаний: правка сбрасывает `indexed`, индексатор обновляет описание по содержимому.
- Создано вручную в вебе: папка без `topic.ipynb` — сервер создаёт шапку-заглушку «без описания» с `indexed = false`; статья без шапки — исправляет индексатор.

## Метаданные секции

Хранятся в `metadata.kb` ячейки. **Главное поле — `indexed`**, остальные вспомогательные.

```json
{"kb": {
  "id": "event-loop",
  "indexed": false,
  "hash": "…",
  "updated": "2026-11-20T14:03:11",
  "changed_by": "mcp",
  "tags": [
    {"text": "event loop", "source": "mcp"},
    {"text": "concurrency", "source": "indexer"}
  ]
}}
```

| Поле | Значение |
|---|---|
| `id` | короткий латинский id (`a-z0-9-`), уникален в статье; якорь пути `статья#id`. У шапки всегда `head`. Родной `id` ячейки Jupyter не используется. |
| `indexed` | `false` — секция изменена и ждёт индексатора; `true` — обработана. Любое изменение текста сбрасывает в `false`. |
| `hash` | первые 16 hex-символов SHA-256 от исходного текста ячейки целиком (с заголовком, UTF-8). Расхождение с фактическим текстом = секцию меняли → `indexed = false`. |
| `updated` | время последнего изменения текста. |
| `changed_by` | `mcp` — потребитель; `web` — человек в вебе Jupyter; `external` — правка мимо Jupyter (найдена по хешу); `indexer` — индексатор. |
| `tags` | теги секции; у шапок пока нет. |

- Нет `id = head` у первой секции статьи или у секции `topic.ipynb` — статья/тема не в порядке, индексатор исправляет.
- Позже: у шапок — темы, hit rate, релевантность; у тегов и секций — релевантность и hit rate.

### id секции
- Потребитель задаёт `id` сам — в пути новой секции (`статья#id`); сервер проверяет формат и уникальность.
- Сохранение в вебе: секции без `id` сервер ставит случайный (например `c7f3a`).
- Индексатор заменяет случайные `id` осмысленными. Старый `id` после этого не работает — «секция не найдена» (индексатор обычно работает, когда базу никто не читает).

### Развитие схемы
Номера версии схемы нет. Новое поколение индекса — новые поля рядом со старыми (например `indexed_v2`); где нового поля нет, сервер считает индекс старым.

## Теги

- Источник тега: `mcp` — потребитель; `indexer` — индексатор; `web` — зарезервировано (пока человек теги не ставит).
- При изменении секции теги не удаляются. Индексатор пересматривает только свои теги (`indexer`).
- Уникальность тегов (в секции и в выдаче), `id` в статье, имён в теме обеспечивает сервер на всех этапах.
- У шапок тегов пока нет.

## Жизненный цикл правок

1. **Старт сервера:** читает все статьи и шапки, строит индексы в памяти. Сверяет `hash` каждой секции; при расхождении — `indexed = false`, `changed_by = external`, `updated = сейчас`. Секции без `kb` получают метаданные и `indexed = false`.
2. **Запись потребителем:** `indexed = false`, новый `hash`, `updated`, `changed_by = mcp`.
3. **Сохранение в вебе Jupyter:** сервер перехватывает сохранение и сверяет `hash` каждой секции: изменённые и новые — `indexed = false`, `changed_by = web`. Ячейки `code`/`raw` исправляются (превращаются в Markdown).
4. **Удаление** секции, статьи, темы: просто исчезает из индекса в памяти.
5. **Одновременная правка** (человек в вебе и MCP): «кто успел — того и тапки». MCP перед каждой записью читает статью и пишет её целиком атомарно. Если потом человек сохранит старую версию из веба — решает Jupyter (предупредит, что файл изменился на диске); сохранил — правка MCP потеряна. Восстановление — из git. Отдельной защиты нет.
6. **Индексатор** берёт статьи, где есть секции с `indexed = false` (от старых `updated` к новым), и обрабатывает **статью целиком**: порядок секций, деление/объединение, `id`, теги `indexer`, описание в шапке; ставит `indexed = true`.

## Версионирование через git

Только на экстренный случай.
- `jupyter_data/kb/` — git-репозиторий. Ведёт его сам сервер через **Dulwich** (git на чистом Python — без бинарника `git`, ставится в `jupyter_bin` вместе с Jupyter).
- **Автокоммит по событию:** первая правка после последнего коммита (MCP, веб, админка) запускает таймер на **3 минуты**; по его истечении — один коммит со всеми правками за это время. Без правок коммитов нет. Плюс коммит при остановке сервера и при старте (правки, сделанные пока сервер не работал).
- Автокоммит работает всегда, в том числе во время работы индексатора; индексатор коммитами не управляет. Сообщение коммита — кто и что менял (`mcp` / `web` / `indexer`, список статей).
- Истории в интерфейсе Jupyter и в API нет. Восстановление — вручную: консольная команда `dulwich` или любой обычный git (репозиторий стандартный).
- Если `ContentsManager` не файловый (нет `root_dir`), автокоммит выключается, остальное работает.
- Задержку можно поменять переменной окружения `KB_MCP_COMMIT_DELAY` (секунды; в тестах — 1).